In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import string

# Initialize random seeds
np.random.seed(42)
random.seed(42)

# Define realistic PM themes from VOC analysis
PAINS = [
    "stakeholder conflicts", "prioritization struggles", "no authority", "endless meetings", 
    "C suite overrides", "decision fatigue", "politics", "no customer validation",
    "roadmap misalignment", "project manager duties"
]

DREAMS = [
    "real user impact", "strategic freedom", "supportive teams", "innovative company",
    "broader role", "better WLB", "own business", "Notion-like culture"
]

PHRASES = [
    "boulder uphill", "CEO without power", "not psychic", "slow down to speed up",
    "C suite idiots", "sandbagging gatekeepers", "impacting users lives"
]

SOURCES = ['Reddit', 'LinkedIn', 'Quora', 'Twitter', 'ProductHunt']
SEGMENTS = ['Enterprise PMs', 'SMB PMs', 'Startup PMs', 'Mid-market PMs']

print("Generating Calendar Dimension (2 years of daily data)...")
# Generate Calendar Dimension (2 years of daily data)
start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 12, 31)
dates = [start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)]

calendar_df = pd.DataFrame({
    'Date': dates,
    'Year': [d.year for d in dates],
    'Quarter': [(d.month - 1) // 3 + 1 for d in dates],
    'Month': [d.month for d in dates],
    'MonthName': [d.strftime('%B') for d in dates],
    'Week': [d.isocalendar()[1] for d in dates],
    'DayOfWeek': [d.strftime('%A') for d in dates],
    'IsCurrentMonth': [d >= datetime(2026, 1, 1) for d in dates]
})

print("Generating Sources Dimension...")
# Generate Sources Dimension
sources_df = pd.DataFrame({
    'SourceID': range(1, len(SOURCES) + 1),
    'SourceName': SOURCES,
    'PlatformType': ['Forum', 'Social', 'Q&A', 'Social', 'Forum']
})

print("Generating Users Dimension (500 unique SaaS PM users)...")
# Generate Users Dimension (500 unique SaaS PM users)
user_ids = [f"USR_{i:04d}" for i in range(1, 501)]  # More readable IDs
users_df = pd.DataFrame({
    'UserID': user_ids,
    'Segment': np.random.choice(SEGMENTS, 500, p=[0.4, 0.25, 0.2, 0.15]),
    'CompanySize': np.random.choice(['1-10', '11-50', '51-200', '201-1000', '1000+'], 500, p=[0.15, 0.2, 0.25, 0.25, 0.15]),
    'PMExperienceYears': np.random.poisson(4, 500).clip(0, 15),
    'ChurnRiskScore': np.random.normal(0.3, 0.15, 500).clip(0, 1).round(3)
})

print("Generating Themes Dimension...")
# Generate Themes Dimension
all_themes = PAINS + DREAMS
themes_df = pd.DataFrame({
    'ThemeID': range(1, len(all_themes) + 1),
    'ThemeName': all_themes,
    'Category': ['Pain'] * len(PAINS) + ['Dream'] * len(DREAMS),
    'ImpactScore': np.random.uniform(0.5, 3.0, len(all_themes)).round(2)
})

print("Generating VOC_Mentions fact table (3000+ rows)...")
# Generate VOC_Mentions Fact Table (>2000 rows total across all facts)
voc_data = []
usage_data = []
revenue_data = []

for i in range(3000):  # Ensure >2000 total rows
    date = np.random.choice(calendar_df['Date'].values)
    source_id = np.random.choice(sources_df['SourceID'].values)
    theme_id = np.random.choice(themes_df['ThemeID'].values)
    user_id = np.random.choice(users_df['UserID'].values)
    
    # Get segment for this user
    segment = users_df[users_df['UserID'] == user_id]['Segment'].iloc[0]
    
    # Logical sentiment based on theme category
    theme_category = themes_df[themes_df['ThemeID'] == theme_id]['Category'].iloc[0]
    sentiment = np.random.normal(-0.4 if theme_category == 'Pain' else 0.6, 0.25)
    sentiment = np.clip(sentiment, -1, 1).round(2)
    
    # Frequency weight (upvotes, replies) - higher for painful themes
    freq_weight = np.random.exponential(5 if theme_category == 'Pain' else 3)
    freq_weight = int(np.clip(freq_weight, 1, 50))
    
    # 30% chance of having exact phrase
    phrase_text = np.random.choice(PHRASES + [None], p=[0.3/len(PHRASES)] * len(PHRASES) + [0.7])
    
    voc_data.append({
        'MentionID': i + 1,
        'Date': date,
        'SourceID': source_id,
        'ThemeID': theme_id,
        'UserID': user_id,
        'Segment': segment,
        'Type': np.random.choice(['Pain', 'Dream', 'Phrase']),
        'SentimentScore': sentiment,
        'FrequencyWeight': freq_weight,
        'PhraseText': phrase_text
    })
    
    # Generate related UsageEvents (70% of VOC mentions have usage)
    if np.random.random() > 0.3:
        feature = np.random.choice(['Stakeholder Tool', 'Roadmap Planner', 'Analytics Dashboard', 'Meeting Optimizer'])
        adoption_rate = np.clip(0.6 + (sentiment * 0.2), 0, 1)
        usage_data.append({
            'EventID': len(usage_data) + 1,
            'Date': date,
            'UserID': user_id,
            'Feature': feature,
            'EventType': np.random.choice(['Adopt', 'Churn'], p=[adoption_rate, 1 - adoption_rate]),
            'ChurnFlag': 1 if np.random.random() < max(0.05, 0.3 - sentiment * 0.2) else 0
        })
    
    # Generate RevenueEvents (20% of VOC mentions impact revenue)
    if np.random.random() > 0.8:
        mrr_amount = np.random.exponential(1500) * (1 + sentiment * 0.1)
        revenue_data.append({
            'RevenueID': len(revenue_data) + 1,
            'Date': date,
            'UserID': user_id,
            'MRRAmount': round(mrr_amount, 2),
            'GrowthMoM': round(np.random.normal(0.05, 0.03), 4)
        })

# Create DataFrames
voc_mentions_df = pd.DataFrame(voc_data)
usage_events_df = pd.DataFrame(usage_data)
revenue_events_df = pd.DataFrame(revenue_data)

print(f"\n Data Generation Complete:")
print(f"   VOC Mentions: {len(voc_mentions_df)} rows")
print(f"   Usage Events: {len(usage_events_df)} rows") 
print(f"   Revenue Events: {len(revenue_events_df)} rows")
print(f"   TOTAL ROWS: {len(voc_mentions_df) + len(usage_events_df) + len(revenue_events_df)}")

# Verify logical relationships
print("\n Sample VOC data (showing sentiment-theme correlation):")
print(voc_mentions_df[['ThemeID', 'SentimentScore', 'FrequencyWeight', 'PhraseText']].head(10))

# Calculate sentiment-churn correlation safely
if len(usage_events_df) > 0:
    merged_data = voc_mentions_df.merge(usage_events_df, on='UserID', how='inner')
    if len(merged_data) > 0 and 'SentimentScore' in merged_data.columns and 'ChurnFlag' in merged_data.columns:
        sentiment_churn_corr = merged_data[['SentimentScore', 'ChurnFlag']].corr().iloc[0, 1]
        print(f"\n📌 Churn correlation with sentiment (should be negative):")
        print(f"   Sentiment-Churn correlation: {sentiment_churn_corr:.3f}")

# Save all tables as CSV files for Power BI import
print("\n Saving CSV files...")
calendar_df.to_csv('Calendar.csv', index=False)
sources_df.to_csv('Sources.csv', index=False)
users_df.to_csv('Users.csv', index=False)
themes_df.to_csv('Themes.csv', index=False)
voc_mentions_df.to_csv('VOC_Mentions.csv', index=False)
usage_events_df.to_csv('UsageEvents.csv', index=False)
revenue_events_df.to_csv('RevenueEvents.csv', index=False)




Generating Calendar Dimension (2 years of daily data)...
Generating Sources Dimension...
Generating Users Dimension (500 unique SaaS PM users)...
Generating Themes Dimension...
Generating VOC_Mentions fact table (3000+ rows)...

✅ Data Generation Complete:
   VOC Mentions: 3000 rows
   Usage Events: 2115 rows
   Revenue Events: 604 rows
   TOTAL ROWS: 5719

📊 Sample VOC data (showing sentiment-theme correlation):
   ThemeID  SentimentScore  FrequencyWeight             PhraseText
0        3           -0.62                3                   None
1        7           -0.39               18                   None
2       17            0.90                1  slow down to speed up
3        6           -0.11                4         C suite idiots
4        8           -0.09                1  impacting users lives
5       10           -0.39                1                   None
6        7           -0.21               28                   None
7       13            0.47               10    